# Streaming large files

`ecv.open` returns a lazy reader that reads only the window you ask for, so files of any size open instantly. This uses the small test recording; the same code works on a file with hundreds of millions of events.

In [1]:
from pathlib import Path

import eventcv as ecv
import numpy as np


# Finds the small test recording in the EventCV repo. Set PATH to your own file instead.
def _fixture(rel="data/test/example.npz"):
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / rel).exists():
            return str(base / rel)
    raise FileNotFoundError(rel)


PATH = _fixture()

reader = ecv.open(PATH, dt_ms=20)  # 20 ms windows
print("windows:", reader.n_slices)
print("span (ms):", reader.time_span_ms, "| duration:", reader.duration_ms)

windows: 3
span (ms): (0.0, 49.718) | duration: 49.718


## Index a window

`slice(n)` returns window `n`, counted from the start of the recording.

In [2]:
frame0 = reader.slice(0)
print(type(frame0).__name__, "with", len(frame0), "events")

EventStream with 54231 events


## Walk every window

`windows()` reads one window at a time.

In [3]:
for i, w in enumerate(reader.windows()):
    print(f"window {i}: {len(w)} events")

window 0: 54231 events
window 1: 37127 events
window 2: 14937 events


## Windows as representations

With `repr=`, each window comes back as a rendered `EventFrame`.

In [4]:
frames = ecv.open(PATH, dt_ms=20, repr="count")
frame = frames.slice(0)
print(type(frame).__name__, "| kind:", frame.kind, "| shape:", frame.numpy().shape)

# The same as rendering the window yourself:
window = ecv.open(PATH, dt_ms=20).slice(0)
print("matches window.count():", np.array_equal(frame.numpy(), window.count().numpy()))

EventFrame | kind: count | shape: (1, 480, 640)
matches window.count(): True


## Slice by time or index

Times are in the file's own time base; this recording starts at 0.

In [5]:
r = ecv.open(PATH)
print("first 10 ms:", len(r.slice(t0_ms=0.0, t1_ms=10.0)), "events")
print("events [0, 1000):", len(r.slice_count(0, 1000)))

first 10 ms: 27044 events
events [0, 1000): 1000


## Remove hot pixels across the recording

`hot_pixel_filter=True` scans the file once and removes stuck pixels from every window. Filtering each window separately is less reliable, because the threshold changes from window to window.

In [6]:
plain = ecv.open(PATH, dt_ms=20)
filtered = ecv.open(PATH, dt_ms=20, hot_pixel_filter=True)

before = sum(len(w) for w in plain.windows())
after = sum(len(w) for w in filtered.windows())
print(f"{before} events -> {after} after hot-pixel removal")

106295 events -> 96323 after hot-pixel removal


## On a large file

The same code, on a 707-million-event HDF5 file:

```python
reader = ecv.open("brisbane_707M.h5", dt_ms=30)  # opens instantly
frame = reader.slice(5000)  # about 32 ms
```

Only the requested window is read, so memory use stays flat. See [Large files](../large-files.md).